In [0]:
import sys, os
import pytz
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/'))

# Install dependecies
packages = ['xgboost',
            'lightgbm',
            'catboost',
            'openpyx1',
            'category_encoders',
            'fsspec']

for package in packages:
    os.system(f"pip install {package}--quiet")
from decorators import *

from sklearn.model_selection import StratifiedKFold, KFold,StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from auxiliary_functions import spark_to_pandas

from sklearn import preprocessing
import matplotlib.pyplot as plt
import pickle
import seaborn as sns from glob
import glob 
import pandas as pd
import numpy as np

%matplotlib inline
import itertools
import warnings
import os
import gc
pd.options.display.max_columns=None
warnings.filterwarnings("ignore")

from pyspark.sql.functions import col, count, date_format
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct
import sys, os
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/')) 

from utils import *
from write_to storage import *
from write_to_storage import pd_read chunks
from pyspark.sql import SparkSession
from catboost import CatBoostClassifier, Pool

In [0]:
###TABLAS UTILIZADAS

lst_tables = [
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccotradeuda',
    'catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variableactivoevolucionclientepyme',
    'catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_clasificacionclientenivelexperienciapyme',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldo',
    'catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemodulodemografico',
    'catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scorepreaprobacionapppyme',
    'catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variablepasivoevolucionsaldopyme',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanal',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientocargopasivo',
    'catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemoduloactivo',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto',
    'catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_matrizmoraponderadaclientemmgr',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientoabonopasivo',
    'catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanalpagotransferencia',
    ]

general_information(lst_tables, n_jobs=4, verbose=True).display()

In [0]:
#OPERACIONES A PARTIR DE 1 MES DE MADURACION
clientes0 = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito")\
    .select("codmes", "codclaveunicocli", "codinternocomputacional", "codclavepartycli")\
        .filter(
            F.trim(F.col("CODPRODUCTO")).isin(
                'CCOPCV', 'CCOEFT', 'CNEEFT', 'PAGPYM', 'CCOALP', 'CCOFRO',
                'CCOCTB', 'CCORHB', 'CCORHC',
                'CCOCTI',
                'CPERLM', 'CPEJLM',
                'CCOEFG', 'CCOEFM', 'CNEEFA', 'CNEEFG', 'CPEEFC', 'CCOEFC', 'DSGEFN',
                'TCRCJD', 'TCRCJS', 'TCRCND', 'TCRCNS', 'TCRNEJ', 'TCRNEN'
            )
            & (F.col("codmes") >= 202306)
            & (F.col("codclaveunicocli").isNotNull())
            & (F.col("codinternocomputacional").isNotNull())
            & (F.col("ctdmesmaduracion") > 0)
        )\
        .distinct()
clientes0.count()

In [0]:
clientes = clientes0\
    .groupBy(
        "codclaveunicocli", "codmes"
    )\
    .agg(
        F.max(F.col("codinternocomputacional")).alias("codinternocomputacional"),
        F.max(F.col("codclavepartycli")).alias("codclavepartycli"),
    )

clientes.count() #319155

In [0]:
ver_paso1=clientes.groupby("codmes","codclaveunicocli").agg(count("*").alias(count))
ver=ver_paso1.filter(F.col("count")>1)
ver.display()

## CONSTRUCCION DE VARIABLES DEL MODELO

In [0]:
CODMES_IN = 202301

bd_MOD_DEMO = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemodulodemografico") \
    .select(F.col("CODMES").alias("CODMES_0"), "codeclaveunicocl", 
            F.col("ctdmesantiguedadempsunat").alias("MOD_DEMO__ctdmesantiguedadempsunat")) \
                .filter(F.col("codmes") >= CODMES_IN) \
                    .distinct()
bd_MOD_DEMO = bd_MOD_DEMO.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
bd_MOD_DEMO = bd_MOD_DEMO.drop("CODMES_0")

bd_MOD_ACT = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemoduloactivo")
   .select(F.col("CODMES_0").alias("CODMES_0"), "codclaveunicocli",
            F.col("pctratiomtodecdeudapymertmtopasivoprmu3m").alias("MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m"),
            F.col("pctratiomtoopeaprmu6mopecprmu12").alias("MOD_ACT__pctratiomtoopeaprmu6mopecprmu12"),
            F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").alias("MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12")
            ) \
                .filter(F.col("codmes") >= CODMES_IN) \
                    .distinct()
bd_MOD_ACT = bd_MOD_ACT.withColumn("codmes", add_codmes_spark("CODMES_0", +1))
bd_MOD_ACT = bd_MOD_ACT.drop("CODMES_0")

bd_APP_SCORE_APROB_PYME = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scorepreaprobacionapppyme")\
    .select(F.col("CODMES").alias("CODMES_0"),
        "codclaveunicocli",
        F.col("atrasomax_crnenr_12_rl").alias("APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl"), 
        F.col("montoade_act_max6_s_hip_rl").alias("APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl"), 
        F.col("utl_3_rl").alias("APP_SCORE_APROB_PYME__utl_3_rl"), 
        F.col("act_eco_fin").alias("APP_SCORE_APROB_PYME__act_eco_fin"), 
        F.col("edad_fin").alias("APP_SCORE_APROB_PYME__edad_fin"), 
        F.col("meses_activo_sf_bu_ma6_0_ag").alias("APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag"), 
        )\
            .filter(F.col("codmes") == 202506)\
                .distinct()
bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.withColumn("codmes", add_codmes_spark("CODMES_0", +1))
bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.drop("CODMES_0")

bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.withColumn("RNG_ACTIVIDAD_ECONOM",
    F.when(F.col("APP_SCORE_APROB_PYME__act_eco_fin").isNull(), 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'PESCA', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'OTROS', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'SERVICIOS', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'ENERGIA', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'CONSTRUCCION', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'ADM_PUBLICA', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'ACT INMOB, EMP Y DE ALQ', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'INDUST_MANUFACT', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'COMERCIO', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'HOGAR', 1)
    .when(F.col("APP_SCORE_APROB_PYME__act_eco_fin") == 'SALUD', 1)
    .otherwise(0)
)

VIDEVAR_MTX_MORA_POND_CLI_MMGR = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_matrizmoraponderadaclientemmgr")\
    .select(F.col("CODMES").alias("CODMES_0"),"codclaveunicocli",
            F.col("mtodeudaclasifriesgofactordsctosolu12").alias("VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12"))\
                .filter(F.col("codmes") == 202505)\
                    .distinct()
VIDEVAR_MTX_MORA_POND_CLI_MMGR = VIDEVAR_MTX_MORA_POND_CLI_MMGR.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
VIDEVAR_MTX_MORA_POND_CLI_MMGR = VIDEVAR_MTX_MORA_POND_CLI_MMGR.drop("CODMES_0")

PASIVO_EVOL_SALD_PYM = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_variablepasivoevolucionsaldopyme")\
    .select(F.col("CODMES").alias("CODMES_0"), "codinternocomputacional",
        F.col("mtoprmincrvariacionmensualprmvigsolu6m").alias("PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m"),
        )\
            .filter(F.col("codmes") == 202505)\
                .distinct()
PASIVO_EVOL_SALD_PYM = PASIVO_EVOL_SALD_PYM.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
PASIVO_EVOL_SALD_PYM = PASIVO_EVOL_SALD_PYM.drop("CODMES_0")

In [0]:
## TABLAS CON DESFASE O DÍA

MTX_RCC_OTRA_DEUDA = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccotradeuda")\
    .select("codmes", "codclaveunicocli",
        F.col("rcc_mto_rdv_max_u3m").alias("MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m")
        )\
        .filter(F.col("codmes") >= CODMES_IN)\
            .distinct()

CLASI_EXPER_CLI = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adr.mmgr_videavariablesmodelos_vu.hm_clasificacionclientenivelexperienciapyme")\
    .select("codmes", "codclaveunicocli",
        F.col("ctdempleado").alias("CLASI_EXPER_CLI__ctdempleado")
        )\
        .filter(F.col("codmes") >= CODMES_IN)\
            .distinct()

EVOL_CLI_PYM = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adr.mmgr_videavariablesmodelos_vu.hm_variableactivoevolucionclientepyme")\
    .select("codmes", "codinternocomputacional",
        F.col("ctdmaxdiamorau6m").alias("EVOL_CLI_PYM__ctdmaxdiamorau6m")
        )\
        .filter(F.col("codmes") >= CODMES_IN)\
            .distinct()

MTX_RESUMEN_SALDO = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldo") \
    .select("codmes", "codclaveunicocli",
        F.col("prod_ctd_sld_act_u1m").alias("MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m"),
        ) \
        .filter(F.col("codmes") >= CODMES_IN) \
            .distinct()

MTX_RESUMEN_ACT_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo")\
    .select("codmes", "codclaveunicocli",
        F.col("prod_mto_sld_fim_pas_min_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24"),
        F.col("prod_mto_sld_fim_tsav_max_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12"),
        F.col("prod_mto_sld_fim_tsav_med_1_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m"),
        F.col("prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m"),
        F.col("prod_mto_sld_prm_tsav_med_1_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m"),
        F.col("prod_pct_pmtsav_pmact_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24"),
        F.col("prod_mto_sld_prm_pas_max_min_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12"),
        F.col("prod_mto_sld_fim_act_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m"),
        F.col("prod_mto_sld_prm_act_max_min_6_6_rt_u6m").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m"),
        F.col("prod_mto_sld_fim_tsav_max_min_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12"),
                )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

MTX_MOV_ABONO_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovimientoabonopasivo")\
    .select("codmes", "codclaveunicocli",
        F.col("isav_tkt_opea_trnf_dol_max_u3m").alias("MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m"),
        )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

MTX_MOV_CARGO_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizmovientocargopasivo")\
    .select("codmes", "codclaveunicocli",
        F.col("isav_tkt_opec_pago_srv_prm_u3m").alias("MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m"),
        )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

MTX_TRX_CANAL_PAGO_TRANSF = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanalpagotransferencia")\
    .select("codmes", "codclaveunicocli",
        F.col("can_ctd_tmo_tot_pag_bcp_frq_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m"),
        F.col("can_mto_tmo_tot_pag_bcp_prm_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m"),
        )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

MTX_RCC_PROD = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto")\
    .select("codmes", "codclaveunicocli",
        F.col("rcc_tip_cond_mor_max_crnor_max_u6m").alias("MTX_RCC_PROD_rcc__tip_cond_mor_max_crnor_max_u6m")
        )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

bd_MTX_TRX_CANAL = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanal")\
    .select("codmes", "codclaveunicocli",
        F.col("can_tkt_tmo_tot_sol_min_u12").alias("MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12")
        )\
            .filter(F.col("codmes") >= CODMES_IN)\
                .distinct()

In [0]:
# Joins por codclaveunicocli
joins_codclave = [
    bd_MOD_DEMO,
    bd_MOD_ACT,
    bd_APP_SCORE_APROB_PYME,
    VIDEVAR_MTX_MORA_POND_CLI_MMGR,
    MTX_RCC_OTRA_DEUDA,
    CLASI_EXPER_CLI,
    MTX_RESUMEN_SALDO,
    MTX_RESUMEN_ACT_PAS,
    MTX_MOV_ABONO_PAS,
    MTX_MOV_CARGO_PAS,
    MTX_TRX_CANAL_PAGO_TRANSF,
    MTX_RCC_PROD,
    bd_MTX_TRX_CANAL
]

# Joins por codinternocomputacional
joins_codinterno = [
    PASIVO_EVOL_SALD_PYM,
    EVOL_CLI_PYM
]

# Paso 1: Joins por codclaveunicocli
result = clientes
for tabla in joins_codclave:
    result = result.join(
        tabla.select('codmes', 'codclaveunicocli', *[c for c in tabla.columns if c not in ['codmes', 'codclaveunicocli']]),
        on=['codmes', 'codclaveunicocli'],
        how='left'
    )

# Paso 2: Joins por codinternocomputacional
for tabla in joins_codinterno:
    result = result.join(
        tabla.select('codmes', 'codinternocomputacional', *[c for c in tabla.columns if c not in ['codmes', 'codinternocomputacional']]),
        on=['codmes', 'codinternocomputacional'],
        how='left'
    )

# base_princ_final = result
# base_princ_final.count()  #319155

In [0]:
variables_incluir = [
    'MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m_cap',
    'CLASI_EXPER_CLI__ctdempleado_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12_cap',
    'EVOL_CLI_PYM__ctdmaxdiamorau6m_cap',
    'MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m',
    'MOD_DEMO__ctdmesantiguedadempsunat_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m',
    'APP_SCORE_APROB_PYME__utl_3_rl_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24_cap',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl',
    'MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m_cap',
    'MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m_cap',
    'MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m',
    'APP_SCORE_APROB_PYME__edad_fin_cap',
    'PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12',
    'MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m_cap',
    'VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12_cap',
    'RNG_ACTIVIDAD_ECONOM',
    'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12_cap',
    'MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m_cap',
    'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12_cap',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m_cap',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m',
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl_cap',
]

In [0]:
feature = [variables_incluir]
len(feature)

In [0]:
GLOB_DUMMY_LIST = [1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -3333333333, 4444444444, 5555555555, 6666666666, 7777777777, -99, -999,
                   4444444444, 5555555555, 6666666666, 7777777777, 1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -99, -3333333333, 44444.4444,
                   555555.5555, 666666.6666, 77777.7777, 111111.1111, -111111.1111, 222222.2222, -222222.2222, 333333.3333, -333333.3333, None]

cols = result.columns
result = (result.select(*[F.when(F.col(c).isin(GLOB_DUMMY_LIST), None)\
                        .otherwise(F.col(c)).alias(c) for c in cols]))

In [0]:
conteo_filas = result.count()

print(f"El total de registros es: {conteo_filas}")

chunk_size = int((conteo_filas / 35 * 0.1)) ## 202306 - 202601
round_size = (10 ** int(np.log10(chunk_size)) // 2)
chunk_size = int(np.ceil(chunk_size / round_size) * round_size)

# Convertir a DataFrame de Pandas
result_codmes = result.select("codmes").distinct()

for codmes in result_codmes.collect():
    try:
        output_path = f'/Workspace/Users/sherylsalazar@bcp.com.pe/02_Proyectos/        06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/bd_universo_pd_bhv_troncal_202506_Valid_Impl/gen_{codmes["codmes"]}'
        os.makedirs(output_path, exist_ok = True)
        print(f"Procesando el codmes: {codmes['codmes']}")
        df_codmes = result.filter(result.codmes == codmes['codmes'])
        pdfdf = spark_to_pandas(df_codmes, batch_size=150, sort_by=['codmes', 'codclaveunicocli'], verbose=True)
        pd_write_in_chunks(
            df=pdfdf,
            directory_path=output_path,
            chunk_size=chunk_size,
            chunk_name='data_partition',
            file_format='parquet',
            verbose=True,
            index=False)
        print(f"Se ha terminado de procesar el codmes: {codmes['codmes']}")
    except Exception as e:
        print(f"Error al procesar el codmes: {codmes['codmes']} debido a {e}")

In [0]:
df = pd_read_chunks(
    directory_path='/Workspace/Users/sherylsalazar@bcp.com.pe/02_Proyectos/    06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/base_202306_202604_PD_32Variables', file_format='parquet',
    verbose=True).sort_values(by=["codmes", "codclaveunicocli"], ascending=[True, True])

In [0]:
for col in df.columns:
    try:
        df[col] = df[col].astype(float)
    except:
        pass

print(df.dtypes)

##TRANSFORMACIONES

In [0]:
### 24 VARIABLES

# 0.999
col12 = "MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m"
df[col12 + "_cap"] = df[col12].clip(upper=54058.633)

# 0.999
col33 = "CLASI_EXPER_CLI__ctdempleado"
df[col33 + "_cap"] = df[col33].clip(lower=0, upper=203.0)

# 0.995
col17 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12"
df[col17 + "_cap"] = df[col17].clip(upper=88040.516)

# 0.999
col35 = "EVOL_CLT_PYM__ctdmaxdiamorau6m"
df[col35 + "_cap"] = df[col35].clip(lower=0, upper=59.0)

# 0.999
col23 = "MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m"
df[col23 + "_cap"] = df[col23].clip(upper=13.0)

# 0.995
col51 = "MOD_DEMO__ctdmestantiguedadempsunat"
df[col51 + "_cap"] = df[col51].clip(upper=396.0)

# 0.995
col32 = "APP_SCORE_APROB_PYME__utl_3_rl"
df[col32 + "_cap"] = df[col32].clip(lower=0, upper=261.9096)

# 0.998
col14 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24"
df[col14 + "_cap"] = df[col14].clip(upper=0.8768666)

# 0.999
col25 = "MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12"
df[col25 + "_cap"] = df[col25].clip(upper= 20916.992)

# 0.998
col21 = "MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24"
df[col21 + "_cap"] = df[col21].clip(upper= 19.579279)

# 0.99
col18 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m"
df[col18 + "_cap"] = df[col18].clip(upper= 4771.6816)

# 0.99
col19 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m"
df[col19 + "_cap"] = df[col19].clip(upper= 4.0419364)

# 0.995
col38 = "MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m"
df[col38 + "_cap"] = df[col38].clip(upper= 480.26755)

col30 = "APP_SCORE_APROB_PYME__edad_fin"
df[col30 + "_cap"] = df[col30].clip(lower=25, upper=75)

# 0.999
col7 = "PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m"
df[col7 + "_cap"] = df[col7].clip(upper= 211663.56)

# 0.998
col16 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m"
df[col16 + "_cap"] = df[col16].clip(upper= 386732.84)

# 0.999
col13 = "MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m"
df[col13 + "_cap"] = df[col13].clip(upper= 469.555)

# 0.995
col1 = "VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12"
df[col1 + "_cap"] = df[col1].clip(upper= 709515.0)

# 0.999
col37 = "MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12"
df[col37 + "_cap"] = df[col37].clip(lower=0, upper=1)

# 0.999
col15 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12"
df[col15 + "_cap"] = df[col15].clip(upper= 372685.25)

# 0.999
col52 = "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m"
df[col52 + "_cap"] = df[col52].clip(upper= 379400.0)

# 0.998
col39 = "MOD_ACT__pctratiomtoopeaprmu6mopecprmu12"
df[col39 + "_cap"] = df[col39].clip(upper= 2.637802)

# 0.999
col26 = "MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m"
df[col26 + "_cap"] = df[col26].clip(upper=112905.56)

# 0.99
col29 = "APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl"
df[col29 + "_cap"] = df[col29].clip(upper=46.0)

# 0.995
# col20 = "MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12"
# df[col20 + "_cap"] = df[col20].clip(upper=3.8366303)

# 0.999
# col19 = "MTX_TRX_POS__pos_tkt_trx_com_sol_prm_p6m"
# df[col19 + "_cap"] = df[col19].clip(lower=0, upper=7833.3335)

# 0.998
# col110 = "MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_sol_g6m"
# df[col110 + "_cap"] = df[col110].clip(upper=133.00755)

# 0.999
# col11 = "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m"
# df[col11 + "_cap"] = df[col11].clip(upper= 46430.434)

# 0.999
# col22 = "MTX_RESUMEN_SALDO__prod_antmin_cvi_prm_u12"
# df[col22 + "_cap"] = df[col22].clip(upper= 318.5)

# 0.999
# col24 = "MTX_RESUMEN_SALDO__prod_mto_sld_pai_g6m"
# df[col24 + "_cap"] = df[col24].clip(upper= 58.04616)

# 0.999
# col8 = "MTX_TRX_POS__pos_tkt_trx_td_prm_u6m"
# df[col8 + "_cap"] = df[col8].clip(lower=0, upper= 6189.8115)

###MODEL

In [0]:
import pandas as pd
import pickle

with open("/Workspace/Users/sheryltsalazar@bcp.com.pe/02_Proyectos/06_Construccion_PD_Troncal_BHV/12_Modelamiento_pt3/Indicadores_Reducc_Prueba_45Noteb/model_v2_REDUCC_32V.pkl", "rb") 
as file:
    model = pickle.load(file)

In [0]:
feature = [
    'MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m_cap',
    'CLASI_EXPER_CLI__ctdempleado_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12_cap',
    'EVOL_CLI_PYM__ctdmaxdiamorau6m_cap',
    'MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m',
    'MOD_DEMO__ctdmesantiguedadempsunat_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m',
    'APP_SCORE_APROB_PYME__utl_3_rl_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24_cap',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl',
    'MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m_cap',
    'MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m_cap',
    'MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m',
    'APP_SCORE_APROB_PYME__edad_fin_cap',
    'PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12',
    'MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m_cap',
    'VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12_cap',
    'RNG_ACTIVIDAD_ECONOM',
    'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12_cap',
    'MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m_cap',
    'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12_cap',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m_cap',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m',
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl_cap',
]

In [0]:
# Preparar tus variables X (features)
# Asegúrate de seleccionar las mismas variables con las que entrenaste el modelo
X = df[feature]

# Predecir la PD
y_pred = model.predict_proba(X)[:, 1]

# Agregar la PD a tu base de datos
df['PD_NUEVO'] = y_pred

# Verificar resultados
# print(df[['PD']].head())
# print(f"\nEstadísticas de PD:")
# print(df['PD'].describe())

In [0]:
from pyspark.sql.functions import expr
import math

df['XB_NUEVO'] - np.log(df['PD_NUEVO'] / (1 - df['PD_NUEVO']))
df['XB_NUEVO_CAL']- df.eval(' -0.7028499957184016 + 0.8303798780817322 * XB_NUEVO')
df['PD_NUEVO_CAL'] - df['XB_NUEVO_CAL'].apply(lambda x: 1/(1+math.exp(-x)))

In [0]:
# df ya es un DataFrame de Pandas
result = df

# Calcular chunk_size
chunk_size = int((len(result) /35* 0.1)) 
round_size = (10**int(np.log10(chunk_size))) // 2
chunk_size = int(np.ceil(chunk_size / round_size) * round_size)

# Obtener los valores únicos de codmes (en Pandas)
result_codmes = result['codmes'].unique()

for codmes in result codmes:
    try
        output_path =f'/Workspace/Users/sherlytsalazar@bcp.com.pe/ 02_Proyectos/06_Construccion_PD_Troncal_BHV/ 15_Actualizar_PD_Hasta202601/base_202306_202604_PD_32Variables_new/gen_{codmes}'
        os.makedirs(output_path, exist_ok=True)
        
        print(f"Procesando el codmes: {codmes}")

        # Filtrar por codmes (en Pandas)
        df_codmes = result[result['codmes']==codmes]

        # Ordenar el DataFrame
        pddf = df_codmes.sort_values(by=['codmes','codclaveunicocli']) I

        # Guardar en chunks usando tu función
        pd_write_in_chunks(df=pddf,
                        directory_path=output_path,
                        chunk_size=chunk_size,
                        chunk_name='data_partition',
                        file_format='parquet',
                        verbose=True,
                        index=False)

        print(f"Se ha terminado de procesar el codmes: {codmes}")

    except Exception as e:
        print(f"Error al procesar el codmes: {codmes} debido a {e}")

In [0]:
df_guard - pd_read_chunks(
    directory_path="/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/ 06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/ base_202306_202604_PD_32Variables_new",file_format-'parquet',
    verbose-True).sort_values(by-["codmes", "codclaveunicocli"], ascending-[True, True])

In [0]:
for col in df_guard.columns:
    try:
        df_guard[col] = df_guard[col].astype(float)
    except:
        pass

print(df_guard.dtypes)

###GUARDAR TABLA

In [0]:
codmes_list = [202306.0,202307.0,202308.0,202309.0,202310.0,202311.0,202312.0,202401.0,202402.0,202403.0,202404.0,202405.0,202406.0,
202407.0,202408.0,202409.0,202410.0,202411.0,202412.0,202501.0,202502.0,202503.0,202504.0,202505.0,202506.0,202507.0,202508.0,202509.0,202510.0,202511.0,202512.0,
202601.0, 202602.0, 202603.0, 202604.0]

for codmes in codmes_list:
    dbutils.fs.cp(
        f"file:/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/
        base_202306_202604_PD_32Variables_new/gen_{codmes}",
        f"abfss://bcp-edv-fabseg@adlscu1lhclbackp05.dfs.core.windows.net/T06086/PD_BHV_CLIENTE_PYME_32Var/base_{codmes}",
        recurse = True
    )

In [0]:
CONS_CONTAINER = "bcp-edv-fabseg"
def read_from_container(path, format = "delta", sep = ","):
    container = f"abfss://{CONS_CONTAINER}@adlscu1lhclbackp05.dfs.core.windows.net/"
    if format == "parquet":
        return spark.read.format("parquet").load(container + path)
    elif format == 'delta':
        return spark.read.format("delta").load(container + path)
    elif format == "csv":
        return spark.read.format('csv').option("delimiter", sep).option("header", True).load(container + path)
    else:
        print("No es formato valido")
        return

In [0]:
df_list = []
for codmes in codmes_list:
    path = f"T06086/PD_BHV_CLIENTE_PYME_32Var/base_{codmes}"
    df = read_from_container(path, format='parquet')
    if df is not None:
        df_list.append(df)
df_spark = df_list[0]
for df in df_list[1:]:
    df_spark = df_spark.unionByName(df)

In [0]:
df_pd_bhv_cli = df_spark\
    .replace(float('nan'), None)

write_to_storage(
    df_pd_bhv_cli,
    tableName='T06086_PD_BHV_CLIENTE_PYME_202306_202604_32Var',
    squad="fabrica",
    location=None,
)